In [32]:
import numpy as np
import matplotlib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
matplotlib.use ("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [33]:
SEED = 42
np.random.seed (SEED)
torch.manual_seed (SEED)

In [34]:
# Synthetic Additive Manufacturing shape dataset
def _base_shape (n_points: int, kind: str):
  if kind == "cube":
    pc = np.random.uniform(-0.5, 0.5, (n_points, 3))
  elif kind == "cylinder":
    theta = np.random.uniform(0, 2 * np.pi, n_points)
    r = np.sqrt(np.random.uniform(0, 0.25, n_points))
    z = np.random.uniform(-0.5, 0.5, n_points)
    pc = np.stack([r * np.cos(theta), r * np.sin(theta), z], axis=1)
  else:
    t = np.random.uniform(0, np.pi, n_points)
    r = np.random.uniform(0.3, 0.5, n_points)
    z = np.random.uniform(-0.2, 0.2, n_points)
    pc = np.stack([r * np.cos(t), r * np.sin(t), z], axis=1)
  return pc.astype(np.float32)

In [35]:
# Synthetic Additive Manufacturing distortion
def _am_distort(pc: np.ndarray, p: np.ndarray):
  layer_t, speed, temp, support, cooling = p
  out = pc.copy()

  # Thermal wraping
  warp = 0.12 * temp * (1.0 - cooling)
  out[:, 0] += warp * out[:, 2] ** 2
  out[:, 1] += warp * 0.4 * out[:, 2]

  # Layer Compression
  out[:, 2] = (1.0 - 0.07 * layer_t)

  # Surface roughness
  noise_std = 0.008 + 0.034 * speed
  out += np.random.normal(0, noise_std, out.shape).astype(np.float32)

  # Support sag
  out[:, 2] -= 0.02 * support
  mask = out[:, 2] < 0
  out[mask, 2] -= 0.04 * (1.0 - support)

  return out

In [36]:
class AMDataset(Dataset):
  def __init__(self, n_samples: int = 2000, n_points: int = 512):
    self.items = []
    shapes = ["cube", "cylinder", "arch"]
    for _ in range(n_samples):
      kind = np.random.choice(shapes)
      nominal = _base_shape(n_points, kind)
      params = np.random.uniform(0.1, 0.9, 5).astype(np.float32)
      distorted = _am_distort(nominal, params)
      self.items.append((nominal, params, distorted))

  def __len__(self):
    return len(self.items)

  def __getitem__(self, i):
    nomi, para, dist = self.items[i]
    return torch.tensor(nomi), torch.tensor(para), torch.tensor(dist)

#PointNet-Autoencoder

In [37]:
class PointNetEncoder(nn.Module):
  def __init__(self, latent_dim: int = 128):
    super().__init__()
    self.conv = nn.Sequential(
        nn.Conv1d(3, 64, 1),     nn.BatchNorm1d(64),   nn.ReLU(),
        nn.Conv1d(64, 128, 1),   nn.BatchNorm1d(128),  nn.ReLU(),
        nn.Conv1d(128, 256, 1),  nn.BatchNorm1d(256),  nn.ReLU(),
        nn.Conv1d(256, 512, 1),  nn.BatchNorm1d(512),  nn.ReLU(),
        nn.Conv1d(512, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU(),
    )
    self.fc = nn.Sequential(
        nn.Linear(1024, 512), nn.BatchNorm1d(512),  nn.ReLU(),
        nn.Linear(512, 256),  nn.BatchNorm1d(256),  nn.ReLU(),
        nn.Linear(256, latent_dim),
    )

def forward(self, x):
  x = x.transpose(2, 1)
  x = self.conv(x)
  x = x.max(dim=2)[0]     #global max pool
  return self.fc(x)

In [38]:
class PointNetDecoder(nn.Module):
  def __init__(self, latent_dim: int = 128, n_points: int = 512):
    super().__init__()
    self.n_points = n_points
    self.net = nn.Sequential(
        nn.Linear(latent_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
        nn.Linear(256, 512), nn.BatchNorm1d(512), nn.ReLU(),
        nn.Linear(512, 1024), nn.BatchNorm1d(1024), nn.ReLU(),
        nn.Linear(1024, n_points * 3),
    )

  def forward(self, z):
    return self.net(z).view(-1, self.n_points, 3)

In [39]:
class PointNetAE(nn.Module):
  def __init__(self, latent_dim: int = 128, n_points: int = 512):
    super().__init__
    self.encoder = PointNetEncoder(latent_dim)
    self.decoder = PointNetDecoder(latent_dim, n_points)

  def forward(self, x):
    return self.decoder(self.encoder(x))

  def encode(self, x):
    return self.encoder(x)

  def decode(self, z):
    return self.decoder(z)

# Geneartive ML model (Latent-GAN)

In [40]:
class LatentGenerator(nn.Module):
  def __init__(self, noise_dim: int = 64, n_params: int = 5, latent_dim: int =128):
    super().__init__()
    self.noise_dim = noise_dim
    self.net = nn.Sequential(
        nn.Linear(noise_dim + n_params, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2),
        nn.Linear(128, 256),                  nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
        nn.Linear(256, 256),                  nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
        nn.Linear(256, latent_dim),
    )

  def forward(self, noise, params):
    return self.net(torch.cat([noise, params], dim=1))

In [41]:
class LatentDiscriminator(nn.Module):
  def __init__(self, latent_dim: int = 128, n_params: int = 5):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(latent_dim + n_params, 256), nn.LeakyReLU(0.2),
        nn.Linear(256, 128), nn.LeakyReLU(0.2),
        nn.Linear(128, 64), nn.LeakyReLU(0.2),
        nn.Linear(64, 1),
    )

  def forward(self, z, params):
    return self.net(torch.cat([z, params], dim=1))

#Evaluation Metrics

In [42]:
def chamfer_distance(pc1: np.ndarray, pc2: np.ndarray):
  diff = pc1[:, None, :] - pc2[None, :, :]
  dist = (diff ** 2).sum(-1)
  return float(dist.min(1).mean() + dist.min(0).mean())

#PointNet-Autoencoder (Training)

In [43]:
def train_autoencoder(ae, loader, epochs=40, lr=1e-3, device="cpu"):
  ae.to(device)
  opt = optim.Adam(ae.parameters(), lr=lr)
  sch = optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.5)
  history = []
  print("\n" + "=" * 55)
  print("PointNet-Autoencoder training")
  print("=" * 55)

  for ep in range(1, epochs + 1):
    ae.train()
    total = 0.0
    for nominal, params, distorted in loader:
      nominal = nominal.to(device)
      params = params.to(device)
      distorted = distorted.to(device)
      recon = ae(distorted)
      diff = recon.unsqueeze(2) - distorted.unsqueeze(1)
      d = (diff ** 2).sum(-1)
      loss = d.min(2)[0].mean() + d.min(1)[0].mean()
      opt.zero_grad(); loss.backward(); opt.step()
      total += loss.item()
    avg = total / len(loader)
    history.append(avg)
    sch.step()
    if ep % 8 == 0 or ep == 1:
      print(f"Epoch {ep:3d}/{epochs} | Chamfer loss: {avg:.6f}")

  print("=" * 55)
  return history

#Generative ML model - GAN (Training)

In [44]:
def train_lgan(ae, G, D, loader, epochs = 50, lr_g = 1e-4, lr_d = 2e-4, noise_dim = 64, device = "cpu"):
  ae.eval()
  G.to(device); D.to(device)
  opt_G = optim.Adam(G.parameters(), lr = lr_g, betas = (0.5, 0.999))
  opt_D = optim.Adam(D.parameters(), lr = lr_d, betas = (0.5, 0.999))
  bce = nn.BCEWithLogitsLoss()
  hist_g, hist_d = [], []

  print("\n" + "=" * 55)
  print("GAN Training")
  print("=" * 55)

  for ep in range(1, epochs + 1):
    G.train(); D.train()
    tot_g = tot_d = 0.0

    for nominal, params, distorted in loader:
      distorted = distorted.to(device)
      params = params.to(device)
      B = distorted.size(0)

      with torch.no_grad():
        z_real = ae.encode(distorted)

      real_lbl = torch.ones(B, 1, device=device)
      fake_lbl = torch.zeros(B, 1, device=device)

      noise = torch.randn(B, noise_dim, device = device)   # discriminator training
      z_fake = G(noise, params).detach()
      loss_D = (bce(D(z_real, params), real_lbl) + bce(D(z_fake, params), fake_lbl))
      opt_D.zero_grad(); loss_D.backward(); opt_D.step()

      noise = torch.randn(B, noise_dim, device = device)   # generator training
      z_fake = G(noise, params)
      loss_G = bce(D(z_fake, params), real_lbl)
      opt_G.zero_grad(); loss_G.backward(); opt_G.step()

      tot_d += loss_D.item()
      tot_g += loss_G.item()

    hist_d.append(tot_d/len(loader))
    hist_g.append(tot_g/len(loader))
    if ep % 10 == 0 or ep == 1:
                print(f" Epoch {ep:3d}/{epochs} | D: {hist_d[-1]:.4f}  G: {hist_g[-1]:.4f}")

  print("=" * 55)
  return hist_g, hist_d

#AI prediction

In [45]:
def predict_distortion(ae, G, nominal_pcs, process_params, noise_dim = 64, n_samples = 1, device = "cpu"):
  ae.eval(); G.eval()
  if isinstance(process_params, np.ndarray):
    process_params = torch.tensor(process_params)
  process_params = process_params.to(device)
  B = process_params.size(0)

  predictions = []
  for _ in range(n_samples):
    noise = torch.randn(B, noise_dim, device = device)
    z_fake = G(noise, process_params)
    pred = ae.decode(z_fake)
    predictions.append(pred.cpu().numpy())

  return predictions

#Visualization

In [46]:
def plot_curves(ae_hist, g_hist, d_hist):
  fig, axes = plt.subplots(1, 2, figsize=(13, 5))

  axes[0].plot(ae_hist, color="green", linewidth = 2)
  axes[0].set_title("AE Chamfer Loss"); axes[0].set_xlabel("Epoch")
  axes[0].grid(True, alpha = 0.3)

  axes[1].plot(d_hist, color = "red", linewidth = 2, label = "discriminator")
  axes[1].plot(g_hist, color = "blue", linewidth = 2, label = "generator")
  axes[1].set_title("GAN Lossess"); axes[1].set_xlabel("Epoch")
  axes[1].legend(); axes[1].grid(True, alpha = 0.3)

  plt.tight_layout()